# TDD-YOLO v2 — tomato disease detection on an RTX 3050 (6 GB)

Trains **TDD-YOLO v2** (fixed CSAF/CAFE + DySample) and a **re-trained TDD-YOLO baseline** with identical
settings, then compares both on the test split against the paper's reported Tomato-Disease numbers.

| change | baseline (authors' code) | v2 |
|---|---|---|
| CSAF | `CBAM_cor` builds its joint MLP inside `forward()`: random, never trained, size depends on H×W | `CSAF2`: every weight trained, nothing depends on H or W |
| CAFE | `co_FEM` calls `branch2_2` twice; `branch2_3` (dilated 3×3) is dead | `CAFE2`: both coordinate-attention branches complete |
| upsampling | `nn.Upsample(nearest)` | `DySample(scale=2, groups=4)` |

Run the cells top to bottom. The training cells can be re-run: they pick up from `runs/<name>/weights/last.pt` if interrupted.

## 1. Environment check

In [ ]:
import os, sys, platform, json, time
from pathlib import Path
import torch, ultralytics
import tddv2  # patches ultralytics parse_model so the yaml files can use the TDD modules

ROOT = Path.cwd()
print(f"python {platform.python_version()} | torch {torch.__version__} | ultralytics {ultralytics.__version__}")
CUDA = torch.cuda.is_available()
if CUDA:
    p = torch.cuda.get_device_properties(0)
    free, total = torch.cuda.mem_get_info()
    print(f"CUDA {torch.version.cuda} | {p.name} | VRAM {total/2**30:.2f} GB total, {free/2**30:.2f} GB free | cuDNN {torch.backends.cudnn.version()}")
else:
    print("WARNING: CUDA not available - training will run on CPU (fine for testing, far too slow for 150 epochs).")
    print("Install the CUDA build of PyTorch, see README.md.")

## 2. Settings
Everything that matters for the comparison is the same for both models. Environment variables of the same name
override the defaults (used by the automated pipeline test; you can ignore them).

In [ ]:
def env(k, d, cast=str):
    v = os.environ.get(k)
    return d if v is None else cast(v)

S = dict(
    imgsz=env("IMGSZ", 640, int),
    batch=env("BATCH", 16, int),          # 16 fits a 6 GB RTX 3050 with AMP; falls back to batch_oom on OOM
    batch_oom=8,
    epochs=env("EPOCHS", 150, int),
    patience=env("PATIENCE", 40, int),
    workers=env("WORKERS", 4, int),
    amp=True,                              # ignored on CPU
    seed=0,
    device=env("DEVICE", 0 if CUDA else "cpu"),
    pretrained=env("PRETRAINED", "yolo11n.pt") or None,   # yolo11n.pt -> layers 0-10 of BOTH models; "" = from scratch
)
DATA_SRC = env("DATA", "data/TD.zip")      # dataset zip or folder
# TD.zip labels its 9 classes c0..c8. Put the real names here (same order as the label ids) if you know them:
CLASS_NAMES = None
SMOKE_FRACTION = env("SMOKE_FRACTION", 0.2, float)
RUN_SMOKE = env("RUN_SMOKE", "1") == "1"
RUN_FULL = env("RUN_FULL", "1") == "1"     # set to "0" to stop after the smoke test

PROJECT = (ROOT / "runs").resolve()
CFG_V2, CFG_BASE = ROOT / "cfg/tdd-yolo-v2.yaml", ROOT / "cfg/tdd-yolo.yaml"
NAME_V2, NAME_BASE = "tdd-yolo-v2", "tdd-yolo-baseline"

# paper: TDD-YOLO on Tomato-Disease (PLOS ONE 2026, doi 10.1371/journal.pone.0334989, text accompanying Table 5)
PAPER = {"P": 0.822, "R": 0.694, "mAP50": 0.797, "mAP50-95": 0.620}
PROJECT.mkdir(exist_ok=True)
print(json.dumps(S, indent=1))

## 3. Dataset
Accepts a zip or a folder with either `images/train` or `train/images` layout. The `data.yaml` inside TD.zip
hard-codes `/content/TD/...` (Colab), so only its class names are used and a fresh yaml with local paths is written.

In [ ]:
from tddv2.data import prepare
DATA = prepare(DATA_SRC, names=CLASS_NAMES)
import yaml; D = yaml.safe_load(open(DATA))
HAS_TEST = "test" in D
print("test split:", "yes" if HAS_TEST else "NO - comparison will fall back to the val split")

## 4. Build check
Both yaml files build; params/GFLOPs at 640; train-mode forward+backward; inference at a second resolution;
`yolo11n.pt` lands in layers 0-10 and nowhere else. `dead_params` lists parameters that receive no gradient:
in the baseline it shows `co_FEM.branch2_3`, the dead branch described above.

In [ ]:
import pandas as pd
from tddv2.build import check_all
build = pd.DataFrame(check_all([CFG_BASE, CFG_V2], weights=S["pretrained"], imgsz=640))
build["params_M"] = (build.params / 1e6).round(3)
build

## 5. Smoke test + time estimate
Each model trains **1 epoch on 20 % of the train split** (validation on the full val split). Callbacks time the
train and validation phases; the projection is `epochs × (train_time / 0.2 + val_time)`. That is an upper bound:
early stopping (`patience`) usually ends the run sooner. Epoch 1 also includes cuDNN autotuning and dataloader warm-up,
so the estimate is slightly pessimistic.

In [ ]:
from tddv2.train import smoke_test
if RUN_SMOKE:
    smoke = pd.DataFrame([smoke_test(c, f"smoke-{n}", DATA, S, PROJECT, SMOKE_FRACTION)
                          for c, n in ((CFG_V2, NAME_V2), (CFG_BASE, NAME_BASE))])
    smoke.to_csv(PROJECT / "smoke_timing.csv", index=False)
    gpu = torch.cuda.get_device_name(0) if CUDA else "CPU"
    print(f"Projected full training on {gpu}: v2 {smoke.projected_h[0]:.2f} h + baseline {smoke.projected_h[1]:.2f} h "
          f"= {smoke.projected_h.sum():.2f} h for {S['epochs']} epochs each (upper bound)")
    display(smoke)

## 6. Train TDD-YOLO v2
Re-run this cell after an interruption (crash, reboot, Ctrl-C): it resumes from `runs/tdd-yolo-v2/weights/last.pt`.

In [ ]:
from tddv2.train import train_or_resume
if RUN_FULL:
    train_or_resume(CFG_V2, NAME_V2, DATA, S, PROJECT)

## 7. Train the TDD-YOLO baseline (authors' modules, same settings)
Same settings, same seed, same backbone init. Resumes the same way.

In [ ]:
if RUN_FULL:
    train_or_resume(CFG_BASE, NAME_BASE, DATA, S, PROJECT)

## 8. Test-split comparison
`best.pt` of each run, evaluated on the **test** split. The paper row is copied from the paper (its own split and
training budget), so the like-for-like comparison is **v2 − re-trained baseline**.

In [ ]:
from tddv2.train import evaluate, run_state
from tddv2.build import summarize
SPLIT = "test" if HAS_TEST else "val"
rows, per_class = [], {}
for name, cfg in ((NAME_V2, CFG_V2), (NAME_BASE, CFG_BASE)):
    w = PROJECT / name / "weights" / "best.pt"
    if not w.exists():
        print(f"{name}: no best.pt yet - train it first"); continue
    if run_state(PROJECT / name) != "done":
        print(f"WARNING {name}: training has not finished; numbers are provisional")
    summ, pc = evaluate(w, DATA, S, split=SPLIT, project=PROJECT / "eval", name=name)
    p, g = summarize(cfg, nc=len(D["names"]), imgsz=S["imgsz"])
    rows.append({"model": name, **summ, "params_M": round(p / 1e6, 3), "GFLOPs": round(g, 2), "split": SPLIT})
    per_class[name] = pc
rows.append({"model": "TDD-YOLO (paper)", **PAPER, "params_M": None, "GFLOPs": None, "split": "paper"})
cmp = pd.DataFrame(rows).set_index("model")
M = ["P", "R", "mAP50", "mAP50-95"]
if {NAME_V2, NAME_BASE} <= set(cmp.index):
    cmp.loc["v2 - baseline"] = (cmp.loc[NAME_V2, M] - cmp.loc[NAME_BASE, M]).tolist() + [None, None, SPLIT]
cmp[M] = cmp[M].astype(float).round(4)
cmp.to_csv(PROJECT / "comparison.csv")
if per_class:
    pcd = pd.DataFrame(per_class).round(4)
    if pcd.shape[1] == 2:
        pcd["v2 - baseline"] = (pcd[NAME_V2] - pcd[NAME_BASE]).round(4)
    pcd.index.name = "class (AP50)"
    pcd.to_csv(PROJECT / "per_class_ap50.csv")
    display(pcd)
print(f"saved {PROJECT/'comparison.csv'} and {PROJECT/'per_class_ap50.csv'}")
cmp

## 9. Training curves

In [ ]:
import matplotlib.pyplot as plt
cols = [("metrics/mAP50(B)", "val mAP50"), ("metrics/mAP50-95(B)", "val mAP50-95"),
        ("train/box_loss", "train box loss"), ("val/box_loss", "val box loss"),
        ("train/cls_loss", "train cls loss"), ("val/cls_loss", "val cls loss")]
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for name in (NAME_V2, NAME_BASE):
    f = PROJECT / name / "results.csv"
    if not f.exists():
        continue
    r = pd.read_csv(f); r.columns = r.columns.str.strip()
    for ax, (c, t) in zip(axes.flat, cols):
        if c in r:
            ax.plot(r["epoch"], r[c], label=name); ax.set_title(t); ax.set_xlabel("epoch")
for ax in axes.flat:
    ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.savefig(PROJECT / "curves.png", dpi=120); plt.show()

## 10. Sample predictions (test split)

In [ ]:
import random
from ultralytics import YOLO
img_dir = Path(D["path"]) / D.get("test", D["val"])
imgs = sorted(p for p in img_dir.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"})
random.Random(0).shuffle(imgs); imgs = imgs[:4]
models = {n: YOLO(PROJECT / n / "weights" / "best.pt") for n in (NAME_V2, NAME_BASE) if (PROJECT / n / "weights" / "best.pt").exists()}
if imgs and models:
    fig, axes = plt.subplots(len(imgs), len(models), figsize=(6 * len(models), 5 * len(imgs)), squeeze=False)
    for i, im in enumerate(imgs):
        for j, (n, m) in enumerate(models.items()):
            r = m.predict(im, imgsz=S["imgsz"], conf=0.25, iou=0.7, device=S["device"], verbose=False)[0]
            axes[i, j].imshow(r.plot()[..., ::-1]); axes[i, j].set_title(f"{n} | {im.name}"); axes[i, j].axis("off")
    plt.tight_layout(); plt.savefig(PROJECT / "samples.png", dpi=100); plt.show()